In [1]:
import pandas as pd
import numpy as np

In [41]:
df=pd.read_csv(r"C:\Users\Anura\OneDrive\Desktop\Projects\Python\startup_funding.csv", encoding="utf-8-sig")
print("Rows, columns:", df.shape)

Rows, columns: (3044, 10)


In [43]:
df.columns= df.columns.str.replace(r"\s+", " ", regex=True).str.strip()
rename_map={
    "sr no":"sr_no",
    "date": "date",
    "startup name":"startup_name",
    "industry vertical":"industry",
    "subvertical":"sub_vertical",
    "city location": "city",
    "investors name": "investors",
    "investmentntype": "investment_type",
    "amount in usd": "amount_usd",
    "remarks": "remarks",
}
df.columns =[rename_map.get(c.lower(), c) for c in df.columns]
print(df.columns.tolist())

['sr_no', 'date', 'startup_name', 'industry', 'sub_vertical', 'city', 'investors', 'investment_type', 'amount_usd', 'remarks']


In [45]:
text_cols = ["startup_name", "industry", "sub_vertical", "city",
             "investors", "investment_type", "remarks"]
junk={
        "\\\\xc2\\\\xa0": " ",
    "\\xc2\\xa0": " ",
    "\\\\xe2\\\\x80\\\\x99": "'",
    "\\xe2\\x80\\x99": "'",
    "\\\\n": " ",
    "\\n": " ",
    "’": "'",
}
for col in text_cols:
    for bad, good in junk.items():
        df[col]=df[col].str.replace(bad, good, regex=False)
        df[col]= df[col].str.strip()

In [25]:
for col in text_cols:
    df[col]= df[col]. replace("Null", np.nan)

In [27]:
df["date"]=pd.to_datetime(df["date"], unit="D", origin="1899-12-30")

In [31]:
df["startup_name"] = df["startup_name"].str.replace("https://www.", "", regex=False)
df["startup_name"] = df["startup_name"].str.replace("\\\\'", "'", regex=False)
df["startup_name"] = df["startup_name"].str.replace("\\'", "'", regex=False)
df["startup_name"] = df["startup_name"].str.strip('"/ ')

In [33]:
df["city"]= df["city"].str.split("/").str[0].str.strip()
df["city"]= df["city"].str.title()
df["city"]= df["city"].replace({
    "Bangalore": "Bengaluru",
    "Gurgaon":"New Delhi",
    "Nw Delhi": "New Delhi",
    "Ahemadabad": "Ahmedabad",
    "Ahemdabad": "Ahmedabad",
    "Kolkatta": "Kolkata",
    "Bhubneswar": "Bhubaneswar"})


In [35]:
df["industry"]=df["industry"]. replace({
    "ECommerce": "eCommerce", "E-Commerce": "eCommerce", "E-commerce": "eCommerce",
    "Ecommerce": "eCommerce", "ecommerce": "eCommerce",
    "Ed-Tech": "EdTech", "Fin-Tech": "FinTech",
    "IT": "Information Technology",
    "Food and Beverage": "Food & Beverage", "Food & Beverages": "Food & Beverage",
    "Food and Beverages": "Food & Beverage"})
    

In [47]:
df["investment_type"]=df["investment_type"].str.lower()
def group_type(x):
    if pd.isna(x):
        return np.nan
    if "pre" in x and "series a" in x:
        return "Pre-Series A"
    if "series" in x: 
        return x.title()
    if "seed" in x or "angel" in x or "angle" in x:
        return "seed/Angel"
    if "debt" in x or "loan" in x:
        return "Debt"
    if "private" in x or "equity" in x:
        return "Private Equity"
    return "Other"

df["investment_type"]= df["investment_type"].apply(group_type)
df["investors"]= df["investors"].str.replace(r"(?i)^undisclosed.*$", "Undisclosed", regex=True)
df["amount_usd"]= pd.to_numeric(df["amount_usd"], errors="coerce")

In [51]:
print("Final shape:", df.shape)
print(df.isnull().sum())
print(df["city"].value_counts().head(8))
print(df["investment_type"].value_counts().head(8))

Final shape: (3044, 10)
sr_no                0
date                 4
startup_name         0
industry           171
sub_vertical       936
city               180
investors            0
investment_type      1
amount_usd         978
remarks             13
dtype: int64
city
Bangalore    701
Mumbai       568
New Delhi    424
Gurgaon      291
Bengaluru    141
Pune         105
Hyderabad     99
Chennai       97
Name: count, dtype: int64
investment_type
seed/Angel        1542
Private Equity    1367
Debt                30
Series A            24
Series B            20
Other               16
Series C            14
Series D            12
Name: count, dtype: int64


In [53]:
df.to_csv(r"C:\Users\Anura\OneDrive\Desktop\Projects\Python\startup_funding_clean.csv", index=False)